# Capture The Flag (CTF) em Criptografia Aplicada

___Objetivo___: Aplicar conceitos práticos de criptografia (Cifras Clássicas, Criptografia Simétrica e Assimétrica, Funções de Resumo e Infraestrutura de Chaves Públicas) em um cenário simulado de análise de artefatos e segurança ofensiva.
## 1. Contexto do Estudo de Caso
---
O presente exercício simula um ambiente de inteligência cibernética.
- Foi disponibilizado a você um pacote de dados correspondente a uma interceptação de rede de uma infraestrutura hostil.
- A análise preliminar indica que este pacote compõe o sistema interno de validação de identidades e transmissão de ordens da referida rede.

O pacote de artefatos contém:
- Um executável responsável pela validação de integridade e confiança (`verificador`).
- Um arquivo em formato binário não reconhecido preliminarmente (`IUVA`).
- Uma nota de texto contendo uma string cifrada (`secret.txt`).

__Diretriz da Atividade__:
- O discente deverá conduzir uma criptoanálise progressiva sobre os artefatos fornecidos para compreender a arquitetura de validação do sistema.
- O objetivo final é explorar o fluxo de execução para forjar um documento, garantindo que sua integridade e sua autenticidade sejam validadas e aceitas pelo executável como legítimas.

# Descrição
---
Precisa-se de detetives
- Nossa equipe de inteligência interceptou um pacote de dados trafegando na rede de um grupo cibercriminoso.
- Acreditamos que o pacote faça parte do processo de integração e compartilhamento do sistema interno de validação de identidades e assinaturas do grupo.

Dados Interceptados
- O pacote interceptado contém
    - um executável responsável pelas validações (verificador.exe),
    - a assinatura desse próprio verificador,
    - um arquivo binário obscuro chamado IUVA
    - e uma nota de texto suspeita chamada secret.txt.
- Sua missão é descobrir como o sistema deles estabelece confiança
- e, o mais importante, elaborar um método para forjar mensagens que o verificador deles aceite como legítimas.

In [ ]:
!pwd
!ls
pkg_path = "./ctf_package/"
!ls -a $pkg_path
print("==================================")

import os, sys

if not os.path.exists(pkg_path):
    print("ctf package not found on", pkg_path)
    sys.exit()
else:
    print("ctf package at", pkg_path)

verifier_exe = f"{pkg_path}verificador"
verifier_sign = f"{pkg_path}assinatura_verificador.bin"
verifier_cert = f"{pkg_path}verificador.pem"
IUVA = f"{pkg_path}IUVA"
note = f"{pkg_path}secret.txt"
root_ca = f"{pkg_path}.trust_store/root_ca.pem"

for f in [verifier_exe, verifier_sign, verifier_cert, IUVA, note, root_ca]:
    if not os.path.exists(f):
        print(f"file {f} not found")
        sys.exit()
    else:
        print(f"Found {f}")

## Passo 1
---
- O único arquivo que conseguimos analisar diretamente é o secret.txt.
- Ele é um arquivo de texto que contém exatamente 64 caracteres quebrado em duas linhas de 32 caracteres.
- Precisamos descobrir como essa informação pode ser útil para os próximos passos da investigação.
- Nossa equipe de criptoanalistas indicam que a mensagem parece utilizar algum tipo de criptografia clássica,
    - mas com uma variação que vai além do alfabeto tradicional (A-Z).
```
Teve$ywev$s$zivmjmgehsv$gsrwypxi$
ew$mrxvyëùiw0$oAwikvihs$EIWcGFG
```
Dica: 64 caracteres ASCII possuem 512 bits

In [ ]:
note_text = ""
with open(note, "r") as f:
    note_text = f.read()

for k in range(-10, 11):
    print(f"{k}: ")
    for c in note_text:
        val = ord(c)
        val -= k
        c = chr(val)
        print(c, end='')
    print()

Valor da cifra de césar: 4

Mensagem decifrada:
```
Para usar o verificador consulte as intruções, k=segredo AES_CBC
```

## Passo 2 
---
Precisamos descobrir o que é esse arquivo para entender as regras do executável.  
- Como podemos quebrar essa camada de segurança?
- Talvez as informações do secret.txt sejam a peça que faltava.

Informações extraídas do arquivo cifrado
- Seguindo com a nossa análise, percebemos que, além do verificador,
- existe um outro arquivo criptografado que pode conter informações vitais sobre o funcionamento da rede inimiga.
- Nossos especialistas em criptoanálise disseram que o cabeçalho do arquivo indica que originalmente era um PDF
    - e que foi utilizada criptografia simétrica moderna com a ferramenta _openssl_.

In [ ]:
import subprocess

IUVA_dec = f"{pkg_path}IUVA_decriptado.pdf"
comando_verify = f"openssl enc -base64 -d -aes-256-cbc -pbkdf2 -in {IUVA} -out {IUVA_dec} -pass pass:segredo"

res_verify = subprocess.run(comando_verify, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not res_verify.returncode == 0:
    print("File not found on", IUVA_dec)
    sys.exit()

if not os.path.exists(IUVA_dec):
    sys.exit()
else:
    print("File at", IUVA_dec)

### Funcionamento obtido no arquivo decriptado

Devido à necessidade de operarmos em ambientes isolados (offline/air-gapped), nosso verificador não se comunica com servidores externos ou Autoridades Certificadoras (CAs) globais. Adotamos um modelo de Confiança Baseada em Diretório Local.

A aprovação de uma assinatura depende exclusivamente de o certificado público do emissor estar previamente catalogado e autorizado na infraestrutura da máquina onde a ferramenta está sendo executada.

___O Processo de Validação (v2.2.0)___

Quando o verificador é executado, ele exige quatro componentes e realiza as seguintes etapas em milissegundos:

| Passo | Camada de Segurança | Ação do Verificador |
| --- | --- | --- |
| 1º | Integridade Rápida | Calcula localmente o HMAC-SHA256 do arquivo da mensagem usando a chave secreta embutida e compara com o hash fornecido. Se falhar, a operação é imediatamente abortada. |
| 2º | Cadeia de Confiança | Varre a pasta ./trust_store/ em busca de uma Autoridade Certificadora (CA) raiz que reconheça e valide o certificado do operador. |
| 3º | Extração | Extrai a chave pública do certificado do operador previamente validado. |
| 4º | Autenticidade | Verifica se a assinatura criptográfica (.bin) condiz com a chave pública extraída e com o arquivo original da mensagem. |

---

___Gestão de Autoridades Certificadoras (CAs)___:
- A arquitetura do verificador foi projetada para ser ágil e de fácil atualização pelos operadores em campo, eliminando a necessidade de recompilar o código-fonte a cada mudança de liderança ou rotação de chaves.

__Como autorizar uma nova entidade (Adicionar nova CA)__:

Se for necessário conceder permissão para que um novo operador emita ordens válidas, siga rigorosamente os passos abaixo:
1. Geração da Chave:
   1. O novo operador deve gerar seu par de chaves e seu certificado público (formato .pem).
2. Inclusão no Repositório:
   1. O operador responsável pela máquina deve copiar o arquivo .pem da nova CA
   2. e colá-lo diretamente na pasta ./trust_store/, localizada no mesmo diretório do executável.
3. Efeito Imediato:
   1. Não há necessidade de reiniciar serviços ou alterar configurações de banco de dados.
   2. A partir do momento em que o certificado .pem estiver fisicamente presente na pasta ./trust_store/, o verificador.exe o assumirá como uma raiz de confiança absoluta.

In [ ]:
# snippet_validacao.py - Núcleo de Processamento do Verificador v2.2.0
import os
import sys
import subprocess
import hmac
import hashlib
def validar_ordem_interna(arquivo_mensagem, arquivo_assinatura, nome_certificado_assinante, hmac_fornecido):
    diretorio_confianca = "./.trust_store/"
    caminho_cert_assinante = f"./{nome_certificado_assinante}"
    print("==================================================")
    print(" VERIFICADOR DE ASSINATURAS E CONFIANÇA - v2.2.0 ")
    print("==================================================")
    # ==========================================
    # PASSO 1: Verificação de Integridade (HMAC)
    # ==========================================
    print("[PASSO 1] Verificando Integridade...")
    try:
        with open(arquivo_mensagem, 'rb') as f:
            mensagem_bytes = f.read()
        hmac_calculado = hmac.new(b"1nt3gr1d4d3", mensagem_bytes, hashlib.sha256).hexdigest()
        if not hmac.compare_digest(hmac_calculado, hmac_fornecido):
            print("[ERRO FATAL] Integridade violada ou código HMAC incorreto. Abortando.")
            sys.exit(1)
        print("[SUCESSO] Código HMAC coincide. O arquivo não foi alterado.")
    except Exception as e:
        print(f"[ERRO] Falha ao processar o arquivo para HMAC: {e}")
        sys.exit(1)

    # ==========================================
    # PASSO 2: Verificação da Cadeia de Confiança
    # ==========================================
    if not os.path.exists(caminho_cert_assinante):
        print(f"[ERRO] Certificado '{nome_certificado_assinante}' não encontrado no diretório atual.")
        sys.exit(1)
    print("\n[PASSO 2] Verificando Cadeia de Confiança do Certificado...")
    ca_confiavel_encontrada = False
    if os.path.exists(diretorio_confianca):
        for ca_file in os.listdir(diretorio_confianca):
            caminho_ca = os.path.join(diretorio_confianca, ca_file)
            if os.path.isfile(caminho_ca):
                # Tenta validar o certificado do assinante contra a CA atual
                comando_verify = f"openssl verify -CAfile {caminho_ca} {caminho_cert_assinante}"
                res_verify = subprocess.run(comando_verify, shell=True,stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                if res_verify.returncode == 0:
                    ca_confiavel_encontrada = True
                    break
        if not ca_confiavel_encontrada:
            print("[ERRO] Operação Negada: A cadeia de confiança do certificado não pôde ser validada por nenhuma CA local.")
            sys.exit(1)

    # ==========================================
    # PASSO 3 e 4: Extração e Verificação
    # ==========================================
    print("\n[PASSO 3] Confiança estabelecida. Extraindo chave pública para verificação da assinatura...")
    comando_pubkey = f"openssl x509 -pubkey -noout -in {caminho_cert_assinante}"
    chave_publica = subprocess.run(comando_pubkey, shell=True, capture_output=True, text=True).stdout
    with open("temp_pubkey.pem", "w") as f:
        f.write(chave_publica)

    print("[PASSO 4] Verificando Assinatura Digital ...")
    comando_dgst = f"openssl dgst -sha256 -verify temp_pubkey.pem -signature {arquivo_assinatura} {arquivo_mensagem}"
    try:
        resultado_dgst = subprocess.run(comando_dgst, shell=True,stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if resultado_dgst.returncode == 0:
            print("\n[SUCESSO] Assinatura Válida. Ordem autêntica e pronta para execução.")
            print("Flag: CTF{Pwned_Trust_Chain_And_HMAC_Bypass}")
        else:
            print("\n[FALHA] A assinatura criptográfica não confere com o arquivo original.")
    except Exception as e:
        print("[ERRO FATAL] Falha na execução do processo openssl.")
    finally:
        if os.path.exists("temp_pubkey.pem"):
            os.remove("temp_pubkey.pem")

## Passo 3
---
Agora que você já sabe como o verificador funciona,
- precisamos descobrir como garantir que os documentos forjados por nós sejam considerados legítimos pela ferramenta.
- Para comprometer a operação do grupo e provar que podemos burlar o sistema deles, você precisará explorar essa falha arquitetural.
- Dica: Não precisamos quebrar a criptografia da assinatura deles.

In [ ]:
import hmac, hashlib, subprocess

# Mensagem forjada
forge_path = f"{pkg_path}forge/"
os.makedirs(forge_path, exist_ok=True)

ca_sign_forge = f"{forge_path}ca_forjada.pem"
ca_key_forge = f"{forge_path}ca_forjada.key"

# Generate the CA private key
comando_gen_ca_key = f"""
openssl genrsa -out {ca_key_forge} 2048
"""
print(subprocess.run(comando_gen_ca_key, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"CA key at: {ca_key_forge}")

# Generate CA signature
comando_cert_sign = f"""
openssl req -new -x509 \
  -key {ca_key_forge} \
  -out {ca_sign_forge} \
  -days 3650 \
  -subj "/CN=CA-Raiz-Falsa"
"""
print(subprocess.run(comando_cert_sign, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"CA signed at: {ca_sign_forge}")

trust_dir = f"{pkg_path}.trust_store/"
os.makedirs(f"{trust_dir}", exist_ok=True)
comando_cp_ca = f"cp {ca_sign_forge} {trust_dir}"
print(subprocess.run(comando_cp_ca, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"CA sign copied")
ca_sign_forge = f"{trust_dir}ca_forjada.pem"

#####################################################
op_forge = f"{forge_path}operador_forjado.key"
op_sign_forge = f"{forge_path}operador_forjado.pem"
op_csr_forge = f"{forge_path}operador_forjado.csr"

# Generate Operator private key
comando_gen_op_key = f"""
openssl genrsa -out {op_forge} 2048
"""
print(subprocess.run(comando_gen_op_key, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"Operator key at: {op_forge}")

# Generate CSR
comando_cert_csr = f"""
openssl req -new \
  -key {op_forge} \
  -out {op_csr_forge} \
  -subj "/CN=Operador"
"""
print(subprocess.run(comando_cert_csr, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"CSR at: {op_csr_forge}")

# Sign CSR with the CA signature
comando_csr_sign = f"""
openssl req -x509 \
  -in {op_csr_forge} \
  -CA {ca_sign_forge} \
  -CAkey {ca_key_forge} \
  -out {op_sign_forge} \
  -days 365
"""
#   -CAcreateserial \
print(subprocess.run(comando_csr_sign, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"CSR signed at: {op_sign_forge}")

# Verify the chain
comando_verify = f"openssl verify -CAfile {ca_sign_forge} {op_sign_forge}"
print(subprocess.run(comando_verify, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"Chain verified")

###################################################
msg_forge = f"{forge_path}mensagem_forjada.txt"
mac_forge = f"{forge_path}hmac_forjado.txt"
sign_forge = f"{forge_path}assinatura_forjada.bin"

# Message
mensagem = "Ordem Forjada: Acesso Total Concedido"
with open(msg_forge, "w") as f:
    f.write(mensagem)

with open(msg_forge, "rb") as f:
    mensagem = f.read()

# HMAC-SHA256
mac = hmac.new(b"1nt3gr1d4d3", mensagem, hashlib.sha256).hexdigest()
with open(mac_forge, "w") as f:
    f.write(mac)
print(f"[+] HMAC : {mac}")

# Assinatura via openssl
comando_sign = f"""
openssl dgst -sha256 \
    -sign {op_forge} \
    -out {sign_forge} \
    {msg_forge}
"""
print(subprocess.run(comando_sign, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE))
print(f"[+] Assinatura gerada: {sign_forge}")

## Passo 4
---
Se tudo estiver certo:
- você deve ser capaz de criar uma identidade se passando por um membro importante do grupo
    - e assinar digitalmente os documentos que quiser.
- Faça uma execução de testes para garantir que tudo está como esperado.

In [ ]:
os.chdir(pkg_path)
verify_command = f"./verificador forge/mensagem_forjada.txt forge/assinatura_forjada.bin forge/operador_forjado.pem {mac}"
subprocess.run(verify_command, shell=True).stdout

## Passo 5
---
O trabalho de um especialista em segurança não termina na quebra do sistema:
- Precisamos catalogar nossos métodos e as vulnerabilidades inimigas no Banco de Dados de Ameaças.
- Sua diretriz final é elaborar um Dossiê Técnico detalhando como a infraestrutura de validação do grupo cibercriminoso foi comprometida.
- Este documento será essencial para subsidiar futuras análises de tráfego e treinamentos da nossa divisão.

O seu relatório deve conter, obrigatoriamente, os seguintes tópicos:
- Vetor de Ataque (Passo a Passo Reprodutível):
    - Documente o fluxo exato da operação,
    - listando e explicando todos os comandos de terminal utilizados desde a análise inicial do secret.txt
    - até a execução final da ferramenta de validação.
- Mapeamento de Primitivas Criptográficas:
    - Catalogue as defesas do grupo inimigo.
    - Identifique e classifique as tecnologias criptográficas que eles empregaram ao longo do processo.
    - Explique brevemente qual era a função de cada uma dentro da arquitetura original deles e como você as contornou.
- Evidências de Comprometimento (IoC):
    - Anexe evidências visuais (capturas de tela) que comprovem a quebra da integridade e da confiança do sistema.
    - É imprescindível incluir o log final do verificador aprovando a sua ordem forjada.

### Diretrizes para Submissão (Relatório Técnico)
---
A avaliação desta atividade está condicionada à entrega de um Relatório Técnico Documental, exclusivamente em formato PDF.

O documento deverá conter obrigatoriamente as seguintes seções:
1. Metodologia e Vetor de Ataque (Reprodutibilidade): Documentação detalhada e sequencial do fluxo da operação. Deve-se listar e fundamentar todos os comandos de terminal executados, desde a inspeção inicial do arquivo `secret.txt` até o acionamento final da ferramenta `verificador`.
2. Mapeamento de Primitivas Criptográficas: Identificação e classificação técnica de todas as tecnologias criptográficas empregadas no sistema alvo. O relatório deve explicar o propósito arquitetural de cada primitiva e o método empregado para contorná-la.
3. Evidenciação de Comprometimento: Anexação de evidências visuais (capturas de tela das saídas de terminal) que comprovem a execução bem-sucedida das etapas.